In [1]:
# Install required packages
!pip install requests beautifulsoup4 nltk textblob vaderSentiment \
             pandas matplotlib seaborn wordcloud plotly -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Red Heart Vinoth\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [3]:
import requests
import json
import re
import time
from datetime import datetime

In [4]:
# ── Core libraries ──────────────────────────────────────────────
import requests
import json
import re
import time
from datetime import datetime
 
# ── Data handling ────────────────────────────────────────────────
import pandas as pd
 
# ── NLP ──────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
 
# ── Web Scraping ─────────────────────────────────────────────────
from bs4 import BeautifulSoup
 
# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
 
print("✅ All imports successful!")

✅ All imports successful!


In [5]:
newsapikey = "53206235b0f04fe48107f78ac5bae4d5"
categories =["business","technology"]
searchqueries =  ["economy","Aritificial Intelligence"]
pagesize = 2
country = "US"
baseurl = "https://newsapi.org/v2"

In [6]:
all_articles =[]

In [7]:
def fetch_top_headlines(categories:str, country:str=country,pagesize:int=pagesize) ->   list:
    url =f"{baseurl}/top-headlines"
    print(url)
    params = {
        "apikey" :   newsapikey,
        "category": categories,
        "counttry" : country,
        "pagesie":pagesize
    } 
    print(params)

    response = requests.get(url,params=params, timeout=10)
    print(response)
    if response.status_code != 200:
        print(f"API error {response.status_code}")

    articles =response.json().get('articles',{})
    print(articles)
    return articles

for cat in categories:
    articles = fetch_top_headlines(categories=cat, country=country)
    for art in articles:
        art["query_category"] = cat
    all_articles.extend(articles)

print(all_articles)

https://newsapi.org/v2/top-headlines
{'apikey': '53206235b0f04fe48107f78ac5bae4d5', 'category': 'business', 'counttry': 'US', 'pagesie': 2}
<Response [200]>
[{'source': {'id': None, 'name': 'Financial Times'}, 'author': 'Leslie Hook', 'title': 'Commodity traders lost ‘billions’ in early days of Iran war - Financial Times', 'description': 'Firms that normally profit from volatility were caught out by sudden rise in energy prices, new report finds', 'url': 'https://www.ft.com/content/3baeeff3-8c7e-4fcd-b704-3ddde3abdbcc?syn-25a6b1a6\\\\u003d1', 'urlToImage': 'https://images.ft.com/v3/image/raw/https%3A%2F%2Fd1e00ek4ebabms.cloudfront.net%2Fproduction%2Fca4473d5-96ab-4afb-91b8-b6a6fc7d2f1d.jpg?source=next-barrier-page', 'publishedAt': '2026-04-12T04:01:56Z', 'content': 'Then $75 per month. Complete digital access to quality FT journalism on any device. Cancel anytime during your trial.'}, {'source': {'id': 'business-insider', 'name': 'Business Insider'}, 'author': 'Lloyd Lee', 'title': 'An

In [8]:
def fetch_by_keyword(query:str,pagesize:int=pagesize) ->   list:
    url = f"{baseurl}/everything"
    print(url)
    request = {
        "apiKey" : newsapikey,
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": pagesize
    }
    print(request)
 

    response = requests.get(url,params=request, timeout=10)
    print(response)
    if response.status_code != 200:
        print(f"API error {response.status_code}")

    articles =response.json().get('articles',{})
    print(articles)

for query in searchqueries:
    articles = fetch_by_keyword(query=query)
    for art in 


SyntaxError: invalid syntax (2069369765.py, line 24)

In [ ]:
def articles_to_dataframe(articles:list) -> pd.DataFrame:
    rows = []
    for art in articles:
        rows.append({
            "title": art.get("title") or "",
            "description" : art.get("description") or "",
            "source" : art.get("source", {}).get("name") or "Unknown",
            "url": art.get("url") or "",
            "published_at" : art.get("publishedAt") or "",
            "category" : art.get("query_category"),
            "full_text" : " ".join(filter(None, [
                art.get("title") or "",
                art.get("description") or ""
            ]))
        })
    df = pd.DataFrame(rows)
    return df
 
df = articles_to_dataframe(all_articles)
df.head()

,title,description,source,url,published_at,category,full_text
0,3.1 million bottles of eye drops sold at Walgr...,The eye drops — sold under multiple brands — h...,CBS News,https://www.cbsnews.com/news/eye-drops-recall-...,2026-04-04T03:57:13Z,business,3.1 million bottles of eye drops sold at Walgr...
1,Anthropic essentially bans OpenClaw from Claud...,The popular combination of OpenClaw and Claude...,The Verge,https://www.theverge.com/ai-artificial-intelli...,2026-04-03T23:52:49Z,business,Anthropic essentially bans OpenClaw from Claud...
2,"Ridership Topped 200,000 on 2 Line Crosslake C...","A total of 205,000 transit riders hopped on li...",Theurbanist.org,https://www.theurbanist.org/ridership-topped-2...,2026-04-03T21:31:40Z,business,"Ridership Topped 200,000 on 2 Line Crosslake C..."
3,Elon Musk insists banks working on SpaceX IPO ...,"Some banks ""agreed to spend tens of millions o...",Ars Technica,https://arstechnica.com/tech-policy/2026/04/el...,2026-04-03T21:17:01Z,business,Elon Musk insists banks working on SpaceX IPO ...
4,SAD! Trump’s AI data center push is failing. B...,Nearly 50% of data center projects delayed as ...,Ars Technica,https://arstechnica.com/tech-policy/2026/04/sa...,2026-04-03T20:43:14Z,business,SAD! Trump’s AI data center push is failing. B...


In [ ]:
# ── Regex patterns used in this project ─────────────────────────
PATTERNS = {
    "url":         re.compile(r'https?://\S+'),
    #before → "Read more at https://bbc.com/news/article-123 today"
    #after  → "Read more at   today"
    "html_tag":    re.compile(r'<[^>]+>'),
    #before → "<b>BREAKING:</b> Markets <em>fall</em> sharply"
    #after  → " BREAKING:  Markets  fall  sharply"
    "punctuation": re.compile(r'[^a-zA-Z\s]'),
    #before → "Apple stock up 3.4% — best day since 2021!"
    #after  → "Apple stock up      best day since      "
    "extra_space": re.compile(r'\s+'),
    #before → "Apple   stock    up      best   day"
    #after  → "Apple stock up best day"
    "ticker":      re.compile(r'\b[A-Z]{2,5}\b'),          # stock tickers / abbreviations
    #input   → "AAPL and TSLA both rose as NASA announced funding"
    #extracts → ["AAPL", "TSLA", "NASA"]
    "proper_noun": re.compile(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+'),  # naive NER
    #input    → "Elon Musk met Tim Cook in New York yesterday"
    #extracts → ["Elon Musk", "Tim Cook", "New York"]
    "year":        re.compile(r'\b(19|20)\d{2}\b'),
    #input    → "The 2024 budget follows the 2023 deficit report"
    #extracts → ["2024", "2023"]
    "number":      re.compile(r'\b\d+\.?\d*\b'),
    #input    → "Stocks fell 4.2 points and GDP grew by 1 percent"
    #extracts → ["4.2", "1"]
    "chars_bracket": re.compile(r'\[\+?\d+\s*chars?\]'),   # NewsAPI truncation marker
    #before → "The economy shrank [+247 chars]"
    #after  → "The economy shrank  "
}
 
def clean_text(text: str) -> str:
    """Full regex cleaning pipeline for NLP."""
    text = PATTERNS["html_tag"].sub(' ', text)        # remove HTML tags
    text = PATTERNS["url"].sub(' ', text)             # remove URLs
    text = PATTERNS["chars_bracket"].sub(' ', text)   # remove [+N chars] artifacts
    text = PATTERNS["punctuation"].sub(' ', text)     # remove punctuation/numbers
    text = PATTERNS["extra_space"].sub(' ', text)     # normalize whitespace
    return text.strip().lower()
 
def extract_proper_nouns(text: str) -> list:
    """Extract likely named entities using a regex heuristic."""
    return PATTERNS["proper_noun"].findall(text)
 
 
def extract_tickers(text: str) -> list:
    """Extract stock ticker / acronym candidates."""
    return PATTERNS["ticker"].findall(text)
 
# ── Apply cleaning to the DataFrame ─────────────────────────────
df["clean_text"]    = df["full_text"].apply(clean_text)
df["proper_nouns"]  = df["full_text"].apply(extract_proper_nouns)
df["tickers"]       = df["title"].apply(extract_tickers)
 
# ── Demo: show before vs after cleaning ─────────────────────────
idx = 1
print("ORIGINAL:\n", df['full_text'].iloc[idx][:300])
print("\nCLEANED:\n",  df['clean_text'].iloc[idx][:300])
print("\nPROPER NOUNS FOUND:", df['proper_nouns'].iloc[idx][:5])

ORIGINAL:
 Anthropic essentially bans OpenClaw from Claude by making subscribers pay extra - The Verge The popular combination of OpenClaw and Claude Code is being severed now that Anthropic has announced it will start charging subscribers extra to access its AI with third-party tools.

CLEANED:
 anthropic essentially bans openclaw from claude by making subscribers pay extra the verge the popular combination of openclaw and claude code is being severed now that anthropic has announced it will start charging subscribers extra to access its ai with third party tools

PROPER NOUNS FOUND: ['The Verge The', 'Claude Code']
